# Script for descriptive analytics / visuals

## Importing dependencies and loading data

In [ ]:
from dependencies import *

In [ ]:
# preprocessed dataset
df_preprocessed = pd.read_csv(preprocessed_csv_filepath, encoding='utf-8-sig')

# pyabsa dataset
df_pyabsa = pd.read_csv(aspect_based_sentiment_csv_filepath, encoding='utf-8-sig')

## Descriptive analytics

### Descriptive analytics for preprocessed dataset contents

In [ ]:
# split dataframes into community groupings
legostarwars_df_preprocessed = df_preprocessed[df_preprocessed['subreddit'].isin(legostarwars_subs)].copy()
lego_df_preprocessed = df_preprocessed[df_preprocessed['subreddit'].isin(lego_subs)].copy()


In [ ]:
# function for computing stats for each split
def compute_stats(df_preprocessed_subset):

    # dataset size
    total_rows = len(df_preprocessed_subset)
    posts = (df_preprocessed_subset['type'] == 'post').sum()
    comments = (df_preprocessed_subset['type'] == 'comment').sum()
    
    comment_texts = df_preprocessed_subset.loc[
        df_preprocessed_subset['type'] == 'comment', 'text'].dropna()

    post_texts = df_preprocessed_subset.loc[
        df_preprocessed_subset['type'] == 'post', 'text'].dropna()

    all_texts = pd.concat([comment_texts, post_texts])
    
    # character length
    char_lengths = all_texts.apply(len)
    avg_char_length = char_lengths.mean() if not char_lengths.empty else 0
    max_char_length = char_lengths.max() if not char_lengths.empty else 0
    min_char_length = char_lengths.min() if not char_lengths.empty else 0
    
    # word count
    word_counts = all_texts.apply(lambda x: len(x.split()))
    avg_word_count = word_counts.mean() if not word_counts.empty else 0
    max_word_count = word_counts.max() if not word_counts.empty else 0
    min_word_count = word_counts.min() if not word_counts.empty else 0
    
    # unique users
    unique_users = df_preprocessed_subset['author'].nunique()

    return {
        'Posts': posts,
        'Comments': comments,
        'Unique users': unique_users,
        'Avg. text length (characters)': avg_char_length,
        'Longest text length (characters)': max_char_length,
        'Shortest text length (characters)': min_char_length,
        'Avg. text length (words)': avg_word_count,
        'Longest text length (words)': max_word_count,
        'Shortest text length (words)': min_word_count}


In [ ]:
def save_stats_tables(name, df_subset, output_prefix):
    stats_tables = {}
    for t in time_periods:
        subset = df_subset[
            df_subset['time_period'].str.contains(t, case=False, na=False)
        ]
        if not subset.empty:
            stats_tables[t] = pd.DataFrame.from_dict(
                compute_stats(subset),
                orient='index',
                columns=[t]
            )
    total_df = pd.DataFrame.from_dict(
        compute_stats(df_subset),
        orient='index',
        columns=['TOTAL']
    )
    stats_tables['TOTAL'] = total_df
    final_table = pd.concat(stats_tables.values(), axis=1)
    row_order = [
        'Posts',
        'Comments',
        'Unique users',
        'Avg. text length (characters)',
        'Longest text length (characters)',
        'Shortest text length (characters)',
        'Avg. text length (words)',
        'Longest text length (words)',
        'Shortest text length (words)'
    ]
    final_table = final_table.reindex(row_order)
    path = f"{output_prefix}_stats.xlsx"
    with pd.ExcelWriter(path, engine="xlsxwriter") as writer:
        final_table.to_excel(writer, sheet_name="stats")
        workbook  = writer.book
        worksheet = writer.sheets["stats"]
        # format: number with comma decimal style (Excel locale-dependent display)
        num_format = workbook.add_format({'num_format': '#,##0.00'})
        # apply formatting to all numeric columns
        for col_num, col_name in enumerate(final_table.columns):
            worksheet.set_column(col_num + 1, col_num + 1, 20, num_format)
    print(f"Saved: {path}")
    return final_table

In [ ]:
# user overlap between community groupings
users_legostarwars = set(legostarwars_df_preprocessed['author'].dropna())
users_lego = set(lego_df_preprocessed['author'].dropna())
unique_to_legostarwars = users_legostarwars - users_lego
unique_to_lego = users_lego - users_legostarwars
overlap_users = users_legostarwars & users_lego
print(f"Unique to legostarwars: {len(unique_to_legostarwars)}")
print(f"Unique to lego: {len(unique_to_lego)}")
print(f"Overlap users: {len(overlap_users)}")

In [ ]:
save_stats_tables("Star Wars", legostarwars_df_preprocessed, "starwars")
save_stats_tables("General", lego_df_preprocessed, "general")
save_stats_tables("Total", df_preprocessed, "total")

### Descriptive analytics for transformed PyABSA dataset

### Descriptive analytics of PyABSA results

In [ ]:
# split dataframe
legostarwars_df_pyabsa = df_pyabsa[df_pyabsa['subreddit'].isin(legostarwars_subs)].copy()
lego_df_pyabsa = df_pyabsa[df_pyabsa['subreddit'].isin(lego_subs)].copy()

In [ ]:
def compute_pyabsa_metrics(df_subset):

    total_rows = len(df_subset)
    
    # average aspects per text
    unique_posts = df_subset['post_id'].dropna().nunique()
    unique_comments = df_subset['comment_id'].dropna().nunique()
    total_texts = unique_posts + unique_comments

    avg_aspects_per_text = (
        total_rows / total_texts
        if total_texts > 0 else 0
    )
    
    # sentiment counts
    neg_count = (df_subset['sentiment_label'] == 'Negative').sum()
    neu_count = (df_subset['sentiment_label'] == 'Neutral').sum()
    pos_count = (df_subset['sentiment_label'] == 'Positive').sum()
    
    # percentages
    def pct(x):
        return (x / total_rows * 100) if total_rows > 0 else 0

    neg_pct = pct(neg_count)
    neu_pct = pct(neu_count)
    pos_pct = pct(pos_count)

    return {
        'Aspect frequencies (rows)': total_rows,
        'Avg. aspect frequency per post/comment': avg_aspects_per_text,
        'Positive labels\n(count & percentage)': (
            f"{pos_count} ({pos_pct:.2f}%)"
        ),
        'Neutral labels\n(count & percentage)': (
            f"{neu_count} ({neu_pct:.2f}%)"
        ),
        'Negative labels\n(count & percentage)': (
            f"{neg_count} ({neg_pct:.2f}%)"
        )
    }

In [ ]:
def save_pyabsa_metrics(name, df_subset, output_prefix):

    metrics_tables = {}

    for t in time_periods:

        subset = df_subset[
            df_subset['time_period'].str.contains(t, case=False, na=False)
        ]

        if not subset.empty:

            metrics = compute_pyabsa_metrics(subset)

            metrics_df = pd.DataFrame.from_dict(
                metrics,
                orient='index',
                columns=[t]
            )

            metrics_tables[t] = metrics_df

    total_metrics = compute_pyabsa_metrics(df_subset)

    total_df = pd.DataFrame.from_dict(
        total_metrics,
        orient='index',
        columns=['TOTAL']
    )

    metrics_tables['TOTAL'] = total_df

    final_table = pd.concat(metrics_tables.values(), axis=1)

    row_order = [
        'Aspect frequencies (rows)',
        'Avg. aspect frequency per post/comment',
        'Positive labels\n(count & percentage)',
        'Neutral labels\n(count & percentage)',
        'Negative labels\n(count & percentage)'
    ]

    final_table = final_table.loc[row_order]

    # format numeric cells only
    def format_cell(x):
        if isinstance(x, (int, float)):
            return f"{x:.2f}".replace(".", ",")
        return x

    final_table = final_table.applymap(format_cell)

    # save Excel
    out_path = f"{output_prefix}_pyabsa_metrics.xlsx"
    final_table.to_excel(out_path)

    print(f"Saved: {out_path}")

    return final_table

In [ ]:
# applying functions
save_pyabsa_metrics(
    "Lego Star Wars communities",
    legostarwars_df_pyabsa,
    "legostarwars")

save_pyabsa_metrics(
    "General LEGO communities",
    lego_df_pyabsa,
    "general_lego")

save_pyabsa_metrics(
    "All subreddits",
    df_pyabsa,
    "all_subreddits")

## Visuals for PyABSA

### Aspect (product quality feature) and keyword proportions & frequencies

In [ ]:
palette = {"t₁": "#1E90FF", "t₂": "#009E73"}


def plot_stacked_bar(ax, df, col, title, xlabel, top_n=None):
    if top_n:
        top = df[col].value_counts().head(top_n).index
        df = df[df[col].isin(top)]
    t1 = df[df["time_period"].str.contains("t₁")][col].value_counts().rename("t₁ (before SMART Brick release)")
    t2 = df[df["time_period"].str.contains("t₂")][col].value_counts().rename("t₂ (after SMART Brick release)")
    counts = pd.concat([t1, t2], axis=1).fillna(0).astype(int)
    counts = counts.loc[counts.sum(axis=1).sort_values(ascending=False).index]

    # strip "_keywords" suffix from index
    counts.index = counts.index.str.replace("_keywords", "", regex=False)

    # convert to percentages of grand total for plotting
    grand_total = counts.values.sum()
    pcts = counts / grand_total * 100

    pcts.plot(kind="bar", stacked=True, ax=ax, color=[palette["t₁"], palette["t₂"]], edgecolor="white", linewidth=0.5)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel("Proportion (%)", fontsize=9)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:.0f}%"))
    ax.tick_params(axis="both", labelsize=8)
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
    ax.legend(title="Time period", fontsize=8, title_fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)

    # annotate with raw counts
    n = len(counts)
    patches = ax.patches
    for i in range(n):
        for j in range(2):
            patch = patches[j * n + i]
            h = patch.get_height()
            raw = counts.iloc[i, j]
            if h > 0:
                ax.text(patch.get_x() + patch.get_width() / 2, patch.get_y() + h / 2,
                        str(raw), ha="center", va="center", fontsize=7, color="black", fontweight="bold")
        total_pct = pcts.iloc[i].sum()
        total_raw = counts.iloc[i].sum()
        ax.text(patches[i].get_x() + patches[i].get_width() / 2, total_pct + ax.get_ylim()[1] * 0.01,
                f"{total_raw}\n({total_pct:.1f}%)", ha="center", va="bottom", fontsize=7, fontweight="bold")

In [ ]:
# creating figure for aspect (product quality feature) frequency
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Aspect (product quality feature) proportions by time period for each community grouping", fontsize=13)
plot_stacked_bar(axes[0], lego_df_pyabsa, "aspect", "General LEGO subreddits", "Aspect (product quality feature)")
plot_stacked_bar(axes[1], legostarwars_df_pyabsa, "aspect", "Niche LEGO Star Wars subreddits", "Aspect (product quality feature)")
plt.tight_layout()
plt.show()

In [ ]:
# creating figure for keyword frequency
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Top 10 keyword proportions by time period for each community grouping", fontsize=13)
plot_stacked_bar(axes[0], lego_df_pyabsa, "keyword_lemmatized", "General LEGO subreddits", "Keyword", top_n=10)
plot_stacked_bar(axes[1], legostarwars_df_pyabsa, "keyword_lemmatized", "Niche LEGO Star Wars subreddits", "Keyword", top_n=10)
plt.tight_layout()
plt.show()

### Sentiment label proportions

In [ ]:
sentiment_palette = {
    "Positive": "#009E73",
    "Neutral":  "#999999",
    "Negative": "#D55E00"}

sentiment_order = ["Positive", "Neutral", "Negative"]

bar_width = 0.43
positions = {
    ("General LEGO subreddits",    "t₁"): 0.0,
    ("General LEGO subreddits",    "t₂"): 0.643,
    ("Niche LEGO Star Wars subreddits",  "t₁"): 1.486,
    ("Niche LEGO Star Wars subreddits",  "t₂"): 2.129}

# prepare data
lego_df_pyabsa["aspect_clean"]         = lego_df_pyabsa["aspect"].str.replace("_keywords", "", regex=False)
legostarwars_df_pyabsa["aspect_clean"] = legostarwars_df_pyabsa["aspect"].str.replace("_keywords", "", regex=False)

aspects = sorted(lego_df_pyabsa["aspect_clean"].unique().tolist())  # alphabetical

n_cols = 3
n_rows = int(np.ceil(len(aspects) / n_cols))  # 4

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4.5 * n_rows), sharey=False)

fig.suptitle("Aspect (product quality feature) sentiment label distribution by community grouping and time period", fontsize=13, y=1.01)
axes_flat = axes.flatten()

# Route the 10th aspect to the centre column of the last row
axes_to_use = list(axes_flat[:9]) + [axes[3][1]]

# Hide unused slots (left and right of last row)
axes[3][0].set_visible(False)
axes[3][2].set_visible(False)

for idx, aspect in enumerate(aspects):
    ax = axes_to_use[idx]

    for (community, period), x_pos in positions.items():
        df = lego_df_pyabsa if "General LEGO" in community else legostarwars_df_pyabsa
        aspect_period_df = df[
            (df["aspect_clean"] == aspect) &
            (df["time_period"].str.contains(period))]
        total = len(aspect_period_df)
        bottom = 0

        for sentiment in sentiment_order:
            count = int((aspect_period_df["sentiment_label"] == sentiment).sum())
            pct = count / total * 100 if total > 0 else 0

            ax.bar(
                x_pos, pct, bar_width,
                bottom=bottom,
                color=sentiment_palette[sentiment],
                edgecolor="white", linewidth=0.5)

            if pct > 0:
                ax.text(
                    x_pos, bottom + pct / 2,
                    f"{count}\n({pct:.1f}%)",
                    ha="center", va="center",
                    fontsize=6, color="black", fontweight="bold",
                    linespacing=1.4)

            bottom += pct

    # vertical dashed separator between communities
    ax.axvline(x=1.065, color="grey", linestyle="--", linewidth=0.8, alpha=0.6)

    ax.set_title(aspect, fontsize=9, fontweight="bold")
    ax.set_xticks(list(positions.values()))
    ax.set_xticklabels(["t₁\n(before SMART\nBrick release)", "t₂\n(after SMART\nBrick release)",
                         "t₁\n(before SMART\nBrick release)", "t₂\n(after SMART\nBrick release)"], fontsize=7)
    ax.set_xlim(-0.35, 2.45)
    ax.set_ylim(0, 115)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0f}%"))
    ax.tick_params(axis="y", labelsize=7)
    ax.spines[["top", "right"]].set_visible(False)

    # community group labels above bars
    ax.text(0.322, 107, "General LEGO subreddits",   ha="center", va="bottom", fontsize=6.5, fontweight="bold", transform=ax.transData)
    ax.text(1.808, 107, "Niche LEGO Star Wars subreddits", ha="center", va="bottom", fontsize=6.5, fontweight="bold", transform=ax.transData)

# legend top right
legend_patches = [mpatches.Patch(color=sentiment_palette[s], label=s) for s in sentiment_order]
fig.legend(handles=legend_patches, title="Sentiment", fontsize=9, title_fontsize=9,
           loc="upper right", bbox_to_anchor=(1.0, 1.0))

plt.tight_layout()
plt.show()